# Module 9: Cryptography and Security

**ACL2 for Computer Science — University-Level Tutorial Series**

## Learning Objectives

By the end of this module you will be able to:

1. Implement and verify modular arithmetic operations in ACL2
2. Understand the mathematical foundations of RSA
3. Model and reason about hash functions formally
4. Appreciate how ACL2 has been used to verify real cryptographic implementations
5. Connect formal verification to security guarantees

## 9.1 Introduction — Why Formal Verification for Cryptography?

Cryptographic algorithms are among the most security-critical software in existence. A single bug can compromise the confidentiality of millions of communications.

Formal verification matters for cryptography because:

- **Subtle bugs are devastating**: Off-by-one errors or incorrect modular reductions can completely break security
- **Testing is insufficient**: Crypto bugs often produce outputs that *look* random and correct but fail against sophisticated attacks
- **Correctness is mathematical**: Crypto algorithms are precisely specified by mathematical definitions — ideal for theorem proving

ACL2 has been used to verify several real cryptographic implementations, including SHA-256, AES, and elliptic curve operations.

### ACL2 Crypto Verification Projects

| Project | Location | What Was Verified |
|---------|----------|-------------------|
| SHA-2 | `books/projects/sha-2/` | SHA-256 and SHA-512 hash functions |
| AES | `books/projects/aes/` | AES block cipher |
| Elliptic Curves | `books/kestrel/crypto/ecurve/` | Elliptic curve operations |
| HMAC | `books/kestrel/crypto/hmac/` | HMAC construction |
| x86 Crypto | `books/projects/x86isa/` | x86 crypto instruction models |

## 9.2 Modular Arithmetic

Modular arithmetic is the foundation of nearly all public-key cryptography. We work with integers modulo some number $n$, where all operations "wrap around" at $n$.

Key identity:

$$a \cdot b \pmod{n} = ((a \bmod n) \cdot (b \bmod n)) \pmod{n}$$

This property is essential for efficient modular exponentiation.

In [ ]:
; Basic modular arithmetic in ACL2.
; ACL2's built-in mod function handles this.
(list (mod 17 5)       ; 2
      (mod 100 7)      ; 2
      (mod (* 3 4) 5)  ; 2
      (mod 0 5))       ; 0

In [ ]:
; The fundamental property of modular multiplication:
; (mod (* a b) n) = (mod (* (mod a n) (mod b n)) n)
(thm
  (implies (and (natp a) (natp b)
                (posp n))
           (equal (mod (* (mod a n) (mod b n)) n)
                  (mod (* a b) n))))

### Modular Exponentiation by Repeated Squaring

Computing $a^e \pmod{n}$ by multiplying $a$ by itself $e$ times would be hopelessly slow for cryptographic-size numbers (e.g., $e > 2^{256}$).

**Repeated squaring** computes this in $O(\log e)$ multiplications using the identity:

$$a^e = \begin{cases} 1 & \text{if } e = 0 \\ (a^{e/2})^2 & \text{if } e \text{ is even} \\ a \cdot a^{e-1} & \text{if } e \text{ is odd} \end{cases}$$

In [ ]:
; Modular exponentiation by repeated squaring
(defun mod-exp (base exp modulus)
  (declare (xargs :guard (and (natp base)
                              (natp exp)
                              (posp modulus))))
  (cond ((zp exp) (mod 1 modulus))
        ((evenp exp)
         (let ((half (mod-exp base (/ exp 2) modulus)))
           (mod (* half half) modulus)))
        (t
         (mod (* base (mod-exp base (- exp 1) modulus))
              modulus))))

In [ ]:
; Test: 2^10 mod 1000 = 1024 mod 1000 = 24
(mod-exp 2 10 1000)

In [ ]:
; Test: 3^13 mod 7
; 3^13 = 1594323, mod 7 = 3
(mod-exp 3 13 7)

In [ ]:
; Verify against direct computation
(equal (mod-exp 5 117 19)
       (mod (expt 5 117) 19))

### The Euclidean Algorithm (GCD)

The **greatest common divisor** is computed efficiently by the Euclidean algorithm and is fundamental to RSA key generation.

In [ ]:
; Euclidean algorithm for GCD
(defun gcd-fn (a b)
  (declare (xargs :guard (and (natp a) (natp b))))
  (if (zp b)
      a
    (gcd-fn b (mod a b))))

In [ ]:
; Tests
(list (gcd-fn 12 8)    ; 4
      (gcd-fn 35 15)   ; 5
      (gcd-fn 17 13)   ; 1 (coprime)
      (gcd-fn 100 0))  ; 100

In [ ]:
; GCD divides both arguments
(thm
  (implies (and (natp a) (posp b))
           (equal (mod a (gcd-fn a b))
                  0))
  :hints (("Goal" :in-theory (enable gcd-fn))))

In [ ]:
; The GCD divides b as well
(thm
  (implies (and (natp a) (posp b))
           (equal (mod b (gcd-fn a b))
                  0))
  :hints (("Goal" :in-theory (enable gcd-fn))))

### Extended Euclidean Algorithm

For RSA, we need the **modular inverse**: given $e$ and $\phi(n)$, find $d$ such that $e \cdot d \equiv 1 \pmod{\phi(n)}$.

The extended Euclidean algorithm finds $x, y$ such that $ax + by = \gcd(a, b)$.

In [ ]:
; Extended Euclidean algorithm
; Returns (list gcd x y) such that a*x + b*y = gcd
(defun ext-gcd (a b)
  (if (zp b)
      (list a 1 0)
    (let* ((result (ext-gcd b (mod a b)))
           (g (first result))
           (x (second result))
           (y (third result)))
      (list g y (- x (* (floor a b) y))))))

In [ ]:
; Test: gcd(35, 15) = 5, and we find x, y such that 35x + 15y = 5
(let ((result (ext-gcd 35 15)))
  (list result
        (+ (* 35 (second result))
           (* 15 (third result)))))  ; should equal 5

In [ ]:
; Modular inverse: find d such that e*d ≡ 1 (mod n)
(defun mod-inverse (e n)
  (let* ((result (ext-gcd e n))
         (d (second result)))
    (mod d n)))

In [ ]:
; Test: inverse of 3 mod 11
; 3 * 4 = 12 ≡ 1 (mod 11)
(mod-inverse 3 11)

## 9.3 RSA Foundations

The **RSA** cryptosystem (Rivest–Shamir–Adleman) is one of the most widely used public-key algorithms. Its security rests on the difficulty of factoring large numbers.

### Key Generation

1. Choose two large primes $p$ and $q$
2. Compute $n = p \cdot q$
3. Compute $\phi(n) = (p-1)(q-1)$
4. Choose $e$ coprime to $\phi(n)$ (typically $e = 65537$)
5. Compute $d = e^{-1} \pmod{\phi(n)}$

**Public key**: $(n, e)$ — **Private key**: $(n, d)$

### Encryption and Decryption

$$\text{encrypt}(m) = m^e \pmod{n}$$
$$\text{decrypt}(c) = c^d \pmod{n}$$

**RSA Correctness Theorem**: $m^{ed} \equiv m \pmod{n}$ for all $0 \le m < n$.

This follows from Euler's theorem: $a^{\phi(n)} \equiv 1 \pmod{n}$ when $\gcd(a, n) = 1$.

In [ ]:
; RSA encryption
(defun rsa-encrypt (message e n)
  (mod-exp message e n))

In [ ]:
; RSA decryption
(defun rsa-decrypt (ciphertext d n)
  (mod-exp ciphertext d n))

### A Small RSA Example

Let's work through RSA with small primes to see the algorithm in action:

In [ ]:
; RSA with small primes for demonstration
; p = 61, q = 53
(let* ((p 61)
       (q 53)
       (n (* p q))          ; 3233
       (phi (* (- p 1) (- q 1)))  ; 3120
       (e 17)               ; coprime to 3120
       (d (mod-inverse e phi)))  ; d such that 17*d ≡ 1 (mod 3120)
  (list :n n
        :phi phi
        :e e
        :d d
        :check-ed-mod-phi (mod (* e d) phi)))  ; should be 1

In [ ]:
; Encrypt and decrypt a message
(let* ((p 61)
       (q 53)
       (n (* p q))
       (phi (* (- p 1) (- q 1)))
       (e 17)
       (d (mod-inverse e phi))
       (message 42)
       (ciphertext (rsa-encrypt message e n))
       (decrypted  (rsa-decrypt ciphertext d n)))
  (list :message message
        :ciphertext ciphertext
        :decrypted decrypted
        :correct (equal decrypted message)))

### The RSA Correctness Theorem

The full proof of RSA correctness requires **Fermat's Little Theorem** (or more generally, Euler's theorem):

$$a^{p-1} \equiv 1 \pmod{p} \quad \text{when } \gcd(a, p) = 1$$

From this, one derives that $m^{ed} = m^{1 + k\phi(n)} = m \cdot (m^{\phi(n)})^k \equiv m \cdot 1^k = m \pmod{n}$.

A full formal proof of this theorem in ACL2 requires significant number theory infrastructure. The `books/projects/numbers/` directory contains relevant foundations.

## 9.4 Hash Functions

A **cryptographic hash function** maps arbitrary-length input to a fixed-length output (the **digest**) with these properties:

1. **Deterministic**: Same input always gives same output
2. **One-way**: Given $h(m)$, it is infeasible to find $m$
3. **Collision-resistant**: Hard to find $m_1 \neq m_2$ with $h(m_1) = h(m_2)$
4. **Avalanche effect**: Small input changes cause large output changes

In [ ]:
; A simple (non-cryptographic!) hash function over lists.
; This is for illustration only — not secure!
(defun simple-hash (lst modulus)
  (if (endp lst)
      0
    (mod (+ (* 31 (simple-hash (cdr lst) modulus))
            (if (natp (car lst)) (car lst) 0))
         modulus)))

In [ ]:
; Test our hash function
(list (simple-hash '(72 101 108 108 111) 1000000)   ; "Hello"
      (simple-hash '(72 101 108 108 112) 1000000)   ; "Hellp" (different)
      (simple-hash '(72 101 108 108 111) 1000000))  ; Same as first (deterministic)

In [ ]:
; Prove determinism: same input → same output
(thm
  (equal (simple-hash lst modulus)
         (simple-hash lst modulus)))

In [ ]:
; A stronger determinism property:
; if two lists are equal, their hashes are equal
(thm
  (implies (equal lst1 lst2)
           (equal (simple-hash lst1 modulus)
                  (simple-hash lst2 modulus))))

In [ ]:
; Our hash always produces a result in [0, modulus)
(thm
  (implies (posp modulus)
           (and (natp (simple-hash lst modulus))
                (< (simple-hash lst modulus) modulus))))

### Real Hash Verification in ACL2

The `books/projects/sha-2/` library contains a formal model of **SHA-256** and **SHA-512** — the hash functions used in TLS, Bitcoin, digital signatures, and countless other applications.

The verification establishes that:
- The ACL2 implementation matches the NIST specification exactly
- The implementation handles all edge cases (padding, length encoding)
- Properties like output size are formally proved

## 9.5 Message Authentication

A **Message Authentication Code (MAC)** allows a recipient to verify both the integrity and authenticity of a message.

### HMAC

**HMAC** (Hash-based MAC) is the standard construction:

$$\text{HMAC}(K, m) = H((K' \oplus \text{opad}) \| H((K' \oplus \text{ipad}) \| m))$$

where $K'$ is the key (padded to block size), and ipad/opad are fixed constants.

In [ ]:
; A simplified MAC scheme for illustration.
; MAC(key, message) = hash(key ++ message)
; (Real HMAC uses the nested construction above)
(defun simple-mac (key message modulus)
  (simple-hash (append key message) modulus))

In [ ]:
; Verify a MAC
(defun verify-mac (key message expected-mac modulus)
  (equal (simple-mac key message modulus)
         expected-mac))

In [ ]:
; Test MAC generation and verification
(let* ((key '(1 2 3))
       (msg '(72 101 108 108 111))
       (mac (simple-mac key msg 1000000)))
  (list :mac mac
        :verify-correct (verify-mac key msg mac 1000000)
        :verify-tampered (verify-mac key '(72 101 108 108 112) mac 1000000)))

In [ ]:
; Correctness: verification always succeeds with the right MAC
(thm
  (verify-mac key message
              (simple-mac key message modulus)
              modulus))

### HMAC in ACL2

The `books/kestrel/crypto/hmac/` library contains formal models of HMAC, verified against the RFC specification.

## 9.6 Verified Cryptography in ACL2

Let's survey the major verified cryptographic projects in ACL2:

### SHA-256 Verification (`books/projects/sha-2/`)

The SHA-2 family (SHA-256, SHA-512) was formally specified and verified in ACL2.

**What was proved:**
- The implementation matches the NIST FIPS 180-4 standard
- Correct handling of message padding and length encoding
- Correct computation of all 64 rounds of compression

**Significance:** SHA-256 is used in TLS, SSH, Bitcoin, git, and digital certificates. Having a verified reference implementation provides high assurance.

### AES Verification (`books/projects/aes/`)

The **Advanced Encryption Standard** (AES) is the most widely used symmetric cipher. The ACL2 verification covers:

- All key sizes (128, 192, 256 bits)
- Key expansion correctness
- Round function correctness (SubBytes, ShiftRows, MixColumns, AddRoundKey)
- Encrypt/decrypt are inverses: $D_k(E_k(m)) = m$

### Elliptic Curve Cryptography (`books/kestrel/crypto/ecurve/`)

Elliptic curve cryptography (ECC) underlies modern key exchange and digital signatures (ECDSA, Ed25519). The ACL2 library verifies:

- Point addition and doubling on elliptic curves
- Curve arithmetic laws (associativity, commutativity)
- Specific curve parameters (e.g., secp256k1 used in Bitcoin)

### x86 Crypto Instructions

The `books/projects/x86isa/` project includes models of x86 instructions like AES-NI, allowing verification that **hardware crypto instructions** compute the correct results.

## 9.7 Exercises

### Exercise 9.1: Modular Exponentiation Properties

Prove that `mod-exp` satisfies these properties:
1. $a^0 \equiv 1 \pmod{n}$ (for $n > 1$)
2. $a^1 \equiv a \pmod{n}$
3. $1^e \equiv 1 \pmod{n}$

In [ ]:
; Exercise 9.1: Prove mod-exp properties

; (a) a^0 ≡ 1 (mod n)
; (thm
;   (implies (and (natp a) (posp n) (> n 1))
;            (equal (mod-exp a 0 n) 1)))

; (b) a^1 ≡ a mod n
; (thm
;   (implies (and (natp a) (posp n))
;            (equal (mod-exp a 1 n) (mod a n))))

; (c) 1^e ≡ 1 mod n
; (thm
;   (implies (and (natp e) (posp n) (> n 1))
;            (equal (mod-exp 1 e n) 1)))

### Exercise 9.2: Primality Testing

Define a simple (trial division) primality tester and verify it classifies some known primes and composites correctly.

In [ ]:
; Exercise 9.2: Primality testing
;
; (defun has-divisor (n d)
;   ;; Check if n has a divisor between 2 and d
;   YOUR DEFINITION)
;
; (defun primep (n)
;   ;; n is prime iff n > 1 and has no divisor from 2 to n-1
;   YOUR DEFINITION)
;
; Tests:
; (primep 2)   → t
; (primep 17)  → t
; (primep 15)  → nil
; (primep 1)   → nil

### Exercise 9.3: Hash Collision Detection

Our `simple-hash` with a small modulus will have many collisions. Write a function that searches for a collision: two different inputs that produce the same hash.

In [ ]:
; Exercise 9.3: Find a hash collision
;
; (defun find-collision (n modulus)
;   ;; Try inputs (list 0), (list 1), ..., (list n)
;   ;; and find two that hash to the same value.
;   YOUR DEFINITION)
;
; With modulus = 100, a collision is guaranteed within 101 inputs
; (by the pigeonhole principle)

### Exercise 9.4: Diffie-Hellman Key Exchange

Model the Diffie-Hellman key exchange protocol:
1. Public parameters: prime $p$, generator $g$
2. Alice picks secret $a$, sends $g^a \bmod p$
3. Bob picks secret $b$, sends $g^b \bmod p$
4. Shared secret: $g^{ab} \bmod p$

Show that both sides compute the same shared secret.

In [ ]:
; Exercise 9.4: Diffie-Hellman
;
; Choose small parameters for testing:
; p = 23, g = 5, a = 6, b = 15
;
; Alice's public value: g^a mod p
; Bob's public value: g^b mod p
; Alice's shared secret: (Bob's public)^a mod p
; Bob's shared secret: (Alice's public)^b mod p
;
; Verify both compute the same shared secret:
; (let* ((p 23) (g 5) (a 6) (b 15)
;        (alice-pub (mod-exp g a p))
;        (bob-pub   (mod-exp g b p))
;        (alice-secret (mod-exp bob-pub a p))
;        (bob-secret   (mod-exp alice-pub b p)))
;   (equal alice-secret bob-secret))

---

**Navigation:**
[< Module 8 — SAT Solving and Boolean Reasoning](08_sat_boolean.ipynb) | [Module 10 — Capstone >](10_capstone.ipynb)